# RNV U-Net training on Colab

Plan and gates: `docs/briefs/colab-parity.md`. Put on Google Drive, under `MyDrive/rnv/`:

- `colab_bundle/`, the output of `run_export` (`cnn/result/DRACTUH_unclustered/colab_bundle/`)
- `colab/`, holding `rnvtorch.py` and `test_parity.py` from the repo

Runtime → Change runtime type → GPU. Train **one fold per session**. Every epoch is checkpointed to Drive, so after a disconnect, rerun the same cell and it resumes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rnv'
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Local disk reads faster than Drive; the bundle is ~620 MB.
!rsync -a {DRIVE}/colab_bundle/ /content/bundle/ && du -sh /content/bundle

**Stages 1–2 (optional, minutes).** Checks that the code and bundle on Drive are the ones that passed locally.

In [ ]:
!python {DRIVE}/colab/test_parity.py /content/bundle

**Stage 3: one-fold smoke run.** Trains fold 1 for 1 epoch, stops, resumes to epoch 2, then writes `P`. It prints seconds per epoch and a projected time for one CV. Before stage 4, multiply that time by this GPU's CU rate: the brief stops at 120 CU for both RNGs together.

In [ ]:
!rm -rf {DRIVE}/runs/smoke
!python {DRIVE}/colab/test_parity.py stage3 /content/bundle {DRIVE}/runs/smoke 2

**Stage 4: parity CV.** Two trainings, `RNG = 0` and `RNG = 1`, folds 1–5, about 8 h each on an L4. `RNG` seeds the random draws (patch list, initial weights, jitter, flips, shuffle, dropout); it is not a localization seed. The cell runs every fold of one RNG. After a disconnect, rerun it: it resumes the unfinished fold, and finished folds only redo their inference (about a minute each). When an RNG is done, download `runs/parity_rng<R>/probcache/` and score it locally:

```matlab
run_cache(cnngvf.defaults(), "<downloaded probcache>")
```

In [ ]:
RNG = 0
for FOLD in range(1, 6):
    !python {DRIVE}/colab/rnvtorch.py train /content/bundle {DRIVE}/runs/parity_rng{RNG} --fold {FOLD} --rng {RNG} --workers 2

**R2: training budget** (`docs/briefs/train-budget.md`). Runs 40 epochs with cosine LR decay, keeps the last epoch, and trains on the inner-validation images. Uses `RNG = 0` only, about 3.3 h per fold on an L4. Rerun the cell after a disconnect: it resumes the unfinished fold. A checkpoint trained with other options refuses to resume. When all 5 folds are done, download `runs/budget_rng0/` (the `probcache/` and every `fold*/log.csv`).

In [ ]:
for FOLD in range(1, 6):
    !python {DRIVE}/colab/rnvtorch.py train /content/bundle {DRIVE}/runs/budget_rng0 --fold {FOLD} --rng 0 --epochs 40 --lr-schedule cosine --output last --no-inner-val --workers 2

**R2 stall retrain.** The brief's stall rule: a fold whose mean training loss over its last five epochs exceeds 0.35 is retrained once from scratch with `--rng 1`, and that retrain is used whatever its score. Fold 3 stalled (0.381). Its retrain goes to its own run root, so it never meets the stalled checkpoint. About 3.4 h. Download `runs/budget_rng0_fold3_rng1/` (`probcache/` and `fold3/log.csv`).

In [ ]:
!python {DRIVE}/colab/rnvtorch.py train /content/bundle {DRIVE}/runs/budget_rng0_fold3_rng1 --fold 3 --rng 1 --epochs 40 --lr-schedule cosine --output last --no-inner-val --workers 2